# Ders 10: Yorumlanabilirlik ve Mekanistik Analiz

**İleri Derin Öğrenme** — Haydar Kılıç

Ön koşul: *Derin Öğrenme*, Ders 10 (Derin öğrenme neden çalışır) ve Ders 4 (Geri yayılım).

Yorumlanabilirlik birbirinden çok farklı iki projeye ayrılır. **Atıf (attribution)**, bir tahmin için
*hangi girdilerin* önemli olduğunu sorar; ucuzdur, popülerdir ve kandırılması kolaydır.
**Mekanistik** yorumlanabilirlik ise içerideki hesabın *ne olduğunu* sorar; pahalıdır ve çok daha
bilgilendiricidir. Bu defterde ikisini de kuruyoruz — bütünleşik gradyanlar ve Shapley değerleri
aksiyomlarıyla, saliency yöntemlerinin çoğunu eleyen akıl sağlığı testleri, doğrusal sondalama ve
aktivasyon yönlendirme ve son olarak süperpozisyon ile onu çözmek için tasarlanan seyrek
otokodlayıcılar.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations

np.random.seed(0)
plt.rcParams["figure.dpi"] = 100
relu = lambda x: np.maximum(x, 0)
sigmoid = lambda x: 1/(1+np.exp(-x))
print("Kütüphaneler yüklendi.")


## 1. Gradyanlar Açıklama Değildir

En basit atıf $\partial f/\partial x_i$'dir: sonsuz küçük bir bozulmada çıktı ne kadar değişirdi?
İki sorun hemen ortaya çıkar.

**Doyma.** Bir öznitelik zaten önemli olmayı bıraktığı noktayı geçmişse — sağlamca açık bir ReLU,
doymuş bir sigmoid — çıktıyı tamamen belirliyor olsa bile yerel gradyanı sıfırdır. Gradyan *katkıyı*
değil *duyarlılığı* ölçer.

**Süreksizlik.** Parçalı doğrusal ağların gradyanları her kırılma noktasında sıçrar; dolayısıyla
harita, modelin gerçek akıl yürütmesiyle hiçbir ilgisi olmayan bir biçimde gürültülüdür.


In [ ]:
# f(x) = 1 - relu(1 - x): çıktı x < 1 için x'e bağlıdır, sonrasında doyar
f = lambda x: 1 - relu(1 - x)
grad = lambda x: (x < 1).astype(float)

xs = np.linspace(0, 2, 400)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
axes[0].plot(xs, f(xs), lw=2, label="f(x)")
axes[0].plot(xs, grad(xs), lw=2, ls="--", label="df/dx")
axes[0].axvline(1.5, c="crimson", lw=1.5)
axes[0].text(1.52, 0.5, "x = 1.5:\nf = 1 ama gradyan = 0", fontsize=9, c="crimson")
axes[0].set_title("Doyma: gradyan katkıyı unutuyor")
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

# 2B bir ağ: gradyan haritası ile bütünlüğe saygılı bir atıf
rng = np.random.default_rng(1)
W1, b1 = rng.normal(size=(2, 12)), rng.normal(size=12)*0.4
W2 = rng.normal(size=12)
net = lambda X: relu(X@W1 + b1) @ W2
def net_grad(X):
    a = (X@W1 + b1) > 0
    return (a*W2) @ W1.T

g = np.stack(np.meshgrid(np.linspace(-2, 2, 120), np.linspace(-2, 2, 120)), -1).reshape(-1, 2)
axes[1].contourf(g[:, 0].reshape(120, 120), g[:, 1].reshape(120, 120), net(g).reshape(120, 120), 25, cmap="RdBu_r")
axes[1].set_title("Ağ çıktısı"); axes[1].set_xticks([]); axes[1].set_yticks([])

gm = np.linalg.norm(net_grad(g), axis=1).reshape(120, 120)
axes[2].imshow(gm, cmap="magma", origin="lower", extent=[-2, 2, -2, 2])
axes[2].set_title("Gradyan büyüklüğü: parçalı sabit, süreksiz")
axes[2].set_xticks([]); axes[2].set_yticks([])
plt.tight_layout(); plt.show()
print("Gradyan haritası, modelin akıl yürütmesinin değil ReLU döşemesinin resmidir.")


## 2. Bütünleşik Gradyanlar ve Bütünlük Aksiyomu

Bütünleşik gradyanlar (IG), doymayı bir **taban çizgisinden** (baseline) $x'$ girdiye $x$ giden
doğrusal bir yol boyunca gradyanları biriktirerek çözer:

$$\text{IG}_i(x) = (x_i - x'_i)\int_0^1 \frac{\partial f\big(x' + \alpha(x-x')\big)}{\partial x_i}\, d\alpha .$$

**Bütünlük** özelliğini sağlar: $\sum_i \text{IG}_i(x) = f(x) - f(x')$ — atıflar tam olarak
açıklanması gereken şeye toplanır. Ayrıca duyarlılık, uygulama değişmezliği ve doğrusallık
aksiyomlarını da sağlar ve (yol doğru olmak kaydıyla) bunu sağlayan tek yöntemdir.

İşin püf noktası **taban çizgisidir**. IG, $f(x) - f(x')$'i açıkladığı için $x'$ seçimi bir uygulama
ayrıntısı değil, sorulan sorunun parçasıdır. Görmede siyah bir görüntü taban çizgisi, "yokluk"u
sessizce "siyah" olarak tanımlar ve koyu pikseller kurgu gereği neredeyse sıfır atıf alır.


In [ ]:
def integrated_gradients(x, baseline, grad_fn, steps=300):
    alphas = (np.arange(steps)+0.5)/steps
    path = baseline + alphas[:, None]*(x - baseline)[None, :]
    return (x - baseline)*grad_fn(path).mean(0)

x  = np.array([1.4, -0.9])
b0 = np.zeros(2)
ig = integrated_gradients(x, b0, net_grad)
print(f"IG atıfları               : {ig.round(4)}")
print(f"atıfların toplamı         : {ig.sum():.5f}")
print(f"f(x) - f(taban çizgisi)   : {float(net(x[None])[0]-net(b0[None])[0]):.5f}   <- bütünlük sağlanıyor")
print(f"düz gradyan*girdi         : {(x*net_grad(x[None])[0]).round(4)}  (toplamı {x@net_grad(x[None])[0]:.4f})")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
alphas = np.linspace(0, 1, 200)
path = b0 + alphas[:, None]*(x-b0)[None, :]
axes[0].plot(alphas, net(path), lw=2)
axes[0].set_xlabel("yol boyunca alpha"); axes[0].set_ylabel("f")
axes[0].set_title("Taban çizgisinden girdiye giden yol"); axes[0].grid(alpha=0.3)

axes[1].plot(alphas, net_grad(path), lw=2)
axes[1].legend(["d f / d x1", "d f / d x2"], fontsize=9)
axes[1].set_xlabel("alpha"); axes[1].set_title("IG, yol boyunca bu gradyanların ortalamasını alır")
axes[1].grid(alpha=0.3)

# Taban çizgisine duyarlılık
bases = {"sıfırlar": np.zeros(2), "birler": np.ones(2), "veri ortalaması": np.array([0.1, -0.2]),
         "rastgele": np.array([-1.5, 1.2])}
vals = np.array([integrated_gradients(x, b, net_grad) for b in bases.values()])
w = 0.35; idx = np.arange(len(bases))
axes[2].bar(idx-w/2, vals[:, 0], w, label="x1'e atıf")
axes[2].bar(idx+w/2, vals[:, 1], w, label="x2'ye atıf")
axes[2].set_xticks(idx); axes[2].set_xticklabels(bases.keys(), fontsize=9)
axes[2].set_title("Taban çizgisi sorunun bir parçasıdır"); axes[2].legend(fontsize=9)
plt.tight_layout(); plt.show()


## 3. Shapley Değerleri: Aksiyomatik Cevap

İşbirlikçi oyun teorisi; etkinlik, simetri, kukla ve doğrusallık aksiyomlarını sağlayan tek atıfı
verir:

$$\phi_i = \sum_{S \subseteq N\setminus\{i\}} \frac{|S|!\,(n-|S|-1)!}{n!}\,\big[v(S \cup \{i\}) - v(S)\big],$$

yani $i$ özniteliğinin tüm sıralamalar üzerinden ortalama marjinal katkısı. Tekliği Shapley
değerlerini cazip kılar; $2^n$ altküme ise pahalı. Bu yüzden pratikteki SHAP bir yaklaşımlar
ailesidir (KernelSHAP ağırlıklı regresyonla, TreeSHAP ağaçlar için tam olarak, DeepSHAP geri
yayılımla).

Az sayıda öznitelikle bunları tam olarak hesaplayıp özellikleri doğrudan görebiliriz — Shapley
değerlerinin **etkileşen öznitelikler arasında payı bölmesi** dahil; bir VE kapısında kredinin
tamamının birine verilmesi yerine her iki girdiye yarım pay düşmesinin nedeni budur.


In [ ]:
from math import factorial

def shapley(value_fn, n):
    phi = np.zeros(n)
    for i in range(n):
        others = [j for j in range(n) if j != i]
        for k in range(len(others)+1):
            for S in combinations(others, k):
                w = factorial(k)*factorial(n-k-1)/factorial(n)
                phi[i] += w*(value_fn(set(S) | {i}) - value_fn(set(S)))
    return phi

# 4 ikili öznitelik üzerinde üç oyuncak "model", her biri farklı etkileşim yapısıyla
n = 4
w_lin = np.array([2.0, -1.0, 0.5, 0.0])
games = {
    "doğrusal (etkileşim yok)": lambda S: sum(w_lin[i] for i in S),
    "0 ve 1 özniteliklerinin VE'si": lambda S: 1.0 if {0, 1} <= S else 0.0,
    "0 ve 1 özniteliklerinin VEYA'sı":  lambda S: 1.0 if ({0} <= S or {1} <= S) else 0.0,
    "0 ve 1 özniteliklerinin XOR'u": lambda S: 1.0 if len({0, 1} & S) == 1 else 0.0,
}

fig, axes = plt.subplots(1, 4, figsize=(17, 3.8))
for ax, (name, v) in zip(axes, games.items()):
    phi = shapley(v, n)
    ax.bar(range(n), phi, color=["#08519c", "#3182bd", "#9ecae1", "#deebf7"])
    ax.axhline(0, c="k", lw=1)
    ax.set_xticks(range(n)); ax.set_xlabel("öznitelik")
    ax.set_title(f"{name}\ntoplam = {phi.sum():.2f}, v(hepsi) = {v(set(range(n))):.2f}", fontsize=10)
plt.suptitle("Tam Shapley değerleri (etkinlik: toplamları daima v(N) - v(boş) eder)", fontsize=12)
plt.tight_layout(); plt.show()

phi_lin = shapley(games["doğrusal (etkileşim yok)"], n)
print("Doğrusallık kontrolü: doğrusal modelin Shapley değerleri ağırlıklarına eşit:", np.allclose(phi_lin, w_lin))
print("Kukla kontrolü      : 3. özniteliğin ağırlığı 0 ve aldığı atıf", f"{phi_lin[3]:.3f}")
phi_and = shapley(games["0 ve 1 özniteliklerinin VE'si"], n)
print("Etkileşim paylaşımı : VE kapısı her girdiye", f"{phi_and[0]:.2f}", "veriyor -- kredi bölünür, çoğaltılmaz")


## 4. Akıl Sağlığı Testleri: Saliency Haritalarının Çoğu Bunları Geçemez

İkna edici görünen bir saliency haritası size modeli değil *görüntüyü* gösteriyor olabilir. Standart
testler (Adebayo vd.) şunlardır:

- **Model rastgeleleştirme.** Ağın ağırlıklarını katman katman rastgeleleştirin. Açıklama neredeyse
  hiç değişmiyorsa, o açıklama modeli açıklamıyordur.
- **Veri rastgeleleştirme.** Etiketleri karıştırarak yeniden eğitin. Gürültüyü ezberlemiş bir model,
  gerçek yapıyı öğrenmiş bir modelle aynı açıklamayı üretmemelidir.

Bazı popüler yöntemler — Guided Backprop bunların başında — bu testlerde ciddi biçimde başarısız olur
ve kenar bulucular gibi davranır. Buradan çıkan ders atıfın değersiz olduğu değil, **bir açıklamanın
bir boş hipoteze karşı doğrulanması gerektiğidir**; tıpkı diğer her ölçüm gibi.


In [ ]:
# Belirgin kenarları olan 1B "görüntü". Gerçek bir gradyan açıklamasıyla kenar-bulucu benzerini karşılaştır.
sig = np.zeros(64); sig[10:20] = 1.0; sig[35:50] = 0.6; sig += 0.03*np.random.randn(64)

def model_grad(x, seed, layers=3, h=32):
    r = np.random.default_rng(seed)
    a, acts = x.copy(), []
    Ws = [r.normal(size=(len(x) if i == 0 else h, h))/np.sqrt(h) for i in range(layers)]
    w_out = r.normal(size=h)/np.sqrt(h)
    for W in Ws:
        z = a@W; acts.append(z > 0); a = relu(z)
    g = w_out
    for W, m in zip(reversed(Ws), reversed(acts)):
        g = (g*m)@W.T
    return g

edge_detector = lambda x: np.abs(np.gradient(x))          # modeli tamamen yok sayar

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
axes[0].plot(sig, lw=2); axes[0].set_title("Girdi sinyali"); axes[0].grid(alpha=0.3)

for seed, c in zip([0, 1, 2, 3], ["#08519c", "#3182bd", "#6baed6", "#c6dbef"]):
    axes[1].plot(np.abs(model_grad(sig, seed)), lw=1.6, c=c, alpha=0.9)
axes[1].set_title("4 FARKLI rastgele model için gradyan saliency\n(değişiyor -> modele bağlı)")
axes[1].grid(alpha=0.3)

for seed, c in zip([0, 1, 2, 3], ["#a50f15", "#de2d26", "#fb6a4a", "#fcae91"]):
    axes[2].plot(edge_detector(sig), lw=1.6, c=c, alpha=0.9)
axes[2].set_title("An edge-detector 'explanation'\n(identical for every model -> explains nothing)")
axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()

grads = np.stack([np.abs(model_grad(sig, s)) for s in range(8)])
corr_model = np.corrcoef(grads)[np.triu_indices(8, 1)].mean()
print(f"farklı rastgele modellerin saliency haritaları arasındaki ortalama korelasyon: {corr_model:.3f}")
print("Rastgeleleştirilmiş modeller arasında korelasyonu ~1.0 olan bir açıklama testi geçememiştir.")


## 5. Sondalama ve Doğrusal Temsil Hipotezi

Mekanistik çalışma bir hipotezle başlar: ağlar, insan için anlamlı kavramları aktivasyon uzayında
**yönler** olarak temsil etme eğilimindedir. Bu doğruysa doğrusal bir sonda kavramı geri kazanabilmeli
ve — asıl sınav — o yön boyunca *müdahale etmek* davranışı değiştirmelidir.

Sondalamanın bilinen bir başarısızlık kipi vardır: yeterince ifade gücü olan bir sonda, modelin
kullanmadığı bilgiyi de çıkarabilir. Buna karşı iki disiplin vardır. **Kontrol görevleri**: aynı
sondayı rastgele etiketlere uydurun; orada da başarılı olan bir sonda kendi kapasitesini ölçüyordur.
**Nedensel müdahale**: aktivasyona $\alpha v$ ekleyin ve çıktının öngörüldüğü gibi değiştiğini
doğrulayın. Yönün gerçekten *kullanıldığını* yalnızca ikincisi gösterir.


In [ ]:
rng = np.random.default_rng(2)
d, n = 32, 3000
v_concept = rng.normal(size=d); v_concept /= np.linalg.norm(v_concept)
v_other   = rng.normal(size=d); v_other -= (v_other@v_concept)*v_concept
v_other  /= np.linalg.norm(v_other)

concept = rng.integers(0, 2, n)*2 - 1                     # gizli ikili kavram
nuisance = rng.normal(size=n)
H = (1.2*concept[:, None]*v_concept + nuisance[:, None]*v_other + 0.5*rng.normal(size=(n, d)))
readout = lambda h: sigmoid(3.0*(h @ v_concept))          # modelin kavramı aşağı akışta kullanımı

def probe(H, y, reg=1e-2):
    w = np.linalg.solve(H.T@H + reg*np.eye(d), H.T@y)
    return w/np.linalg.norm(w)

w_real = probe(H, concept.astype(float))
w_ctrl = probe(H, rng.integers(0, 2, n)*2.0 - 1)          # kontrol görevi: rastgele etiketler

acc_real = np.mean(np.sign(H@w_real) == concept)
y_ctrl = rng.integers(0, 2, n)*2 - 1
acc_ctrl = np.mean(np.sign(H@probe(H, y_ctrl.astype(float))) == y_ctrl)
print(f"gerçek kavramda sonda doğruluğu : {acc_real:.3f}")
print(f"kontrol görevinde sonda doğruluğu: {acc_ctrl:.3f}   (seçicilik = {acc_real-acc_ctrl:.3f})")
print(f"kosinüs(geri kazanılan yön, gerçek yön) = {abs(w_real@v_concept):.3f}")

alphas = np.linspace(-3, 3, 61)
steer_real = [readout(H + a*w_real).mean() for a in alphas]
steer_ctrl = [readout(H + a*w_ctrl).mean() for a in alphas]
steer_rand = [readout(H + a*(lambda u: u/np.linalg.norm(u))(rng.normal(size=d))).mean() for a in alphas]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
axes[0].hist(H@w_real, bins=50, alpha=0.7, label="hepsi")
axes[0].hist((H@w_real)[concept > 0], bins=50, alpha=0.7, label="kavram = +1")
axes[0].set_xlabel("sonda yönüne izdüşüm"); axes[0].set_title("Kavram doğrusal olarak ayrılabilir")
axes[0].legend(fontsize=9)

axes[1].plot(alphas, steer_real, lw=2, label="sonda yönü boyunca yönlendir")
axes[1].plot(alphas, steer_ctrl, lw=2, label="kontrol yönü boyunca yönlendir")
axes[1].plot(alphas, steer_rand, lw=2, label="rastgele bir yön boyunca yönlendir")
axes[1].set_xlabel("müdahale şiddeti alpha"); axes[1].set_ylabel("ortalama model çıktısı")
axes[1].set_title("Nedensel test: bu yön bir ŞEY yapıyor mu?")
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

sizes = [30, 60, 125, 250, 500, 1000, 3000]
sel = []
for m in sizes:
    a1 = np.mean(np.sign(H[:m]@probe(H[:m], concept[:m].astype(float))) == concept[:m])
    a2 = np.mean(np.sign(H[:m]@probe(H[:m], y_ctrl[:m].astype(float))) == y_ctrl[:m])
    sel.append(a1-a2)
axes[2].semilogx(sizes, sel, "o-", lw=2)
axes[2].set_xlabel("sonda eğitim kümesi boyutu"); axes[2].set_ylabel("seçicilik (gerçek - kontrol)")
axes[2].set_title("Küçük sondalama kümeleri kodlananı abartır"); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 6. Süperpozisyon: Nöronlar Neden Çok Anlamlı?

Ağların sıklıkla, sahip oldukları boyut sayısından daha fazla özniteliği temsil etmesi gerekir.
Öznitelikler **seyrek** olduğunda — aynı anda yalnızca birkaçı aktifken — ağ, $m > d$ tanesini
*neredeyse* dik yönler olarak saklayabilir ve karşılığında az miktarda girişimi kabul eder. Buna
**süperpozisyon** denir ve tek tek nöronların birbiriyle alakasız şeylere tepki vermesinin nedeni
budur.

Oyuncak model (Elhage vd.) doğrusal bir otokodlayıcıya bir doğrusalsızlık eklenmiş hâlidir:

$$\hat x = \text{ReLU}\big(W^\top W x + b\big), \qquad W \in \mathbb{R}^{d \times m},\ m > d .$$

Yoğun özniteliklerle model yalnızca en iyi $d$ tanesini tutup gerisini atabilir. Seyreklik arttıkça
fazladan öznitelikleri aynı $d$ boyuta paketlemeye başlar.


In [ ]:
def train_toy(m=8, d=2, sparsity=0.0, steps=6000, lr=0.05, seed=0, batch=512):
    rng = np.random.default_rng(seed)
    imp = 0.85**np.arange(m)                                # öznitelik önemi
    W = rng.normal(size=(d, m))*0.3
    b = np.zeros(m)
    for _ in range(steps):
        x = rng.random((batch, m))*(rng.random((batch, m)) > sparsity)
        h = x @ W.T
        xr = relu(h @ W + b)
        e = 2*(xr - x)*imp*(xr > 0)                     # dL/d(ön-aktivasyon)
        gW = h.T @ e + (e @ W.T).T @ x                  # W iki kez geçer: kodlayıcı ve kod çözücü
        W -= lr*gW/batch
        b -= lr*e.mean(0)
    return W, imp

fig, axes = plt.subplots(1, 4, figsize=(17, 4.2))
sparsities = [0.0, 0.7, 0.9, 0.99]
for ax, s in zip(axes, sparsities):
    W, imp = train_toy(sparsity=s)
    norms = np.linalg.norm(W, axis=0)
    for i in range(W.shape[1]):
        ax.plot([0, W[0, i]], [0, W[1, i]], lw=2.2, alpha=0.85)
    th = np.linspace(0, 2*np.pi, 200); ax.plot(np.cos(th), np.sin(th), c="k", lw=0.6, alpha=0.3)
    ax.set_aspect("equal"); ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"seyreklik {s:.2f}\n8 özniteliğin {(norms > 0.15).sum()} tanesi 2 boyutta temsil ediliyor", fontsize=10)
plt.suptitle("Süperpozisyon oyuncak modeli: seyrek öznitelikler çok az boyuta paketleniyor", fontsize=13)
plt.tight_layout(); plt.show()

for s in sparsities:
    W, _ = train_toy(sparsity=s)
    G = W.T@W; off = np.abs(G - np.diag(np.diag(G)))
    print(f"seyreklik {s:.2f}: tutulan öznitelik {(np.linalg.norm(W,axis=0) > 0.15).sum()}/8, "
          f"ortalama girişim {off.mean():.3f}")


## 7. Seyrek Otokodlayıcılar: Süperpozisyonu Çözmek

Öznitelikler $d$ boyutlu bir aktivasyonda süperpoze edilmişse, onları **aşırı tam (overcomplete) ve
seyrek** bir sözlük öğrenerek geri kazanmayı deneyebiliriz:

$$z = \text{ReLU}(W_e (a - b_d) + b_e), \qquad \hat a = W_d z + b_d,
\qquad \mathcal{L} = \lVert a - \hat a\rVert^2 + \lambda \lVert z\rVert_1 .$$

Gizli katman aktivasyondan çok daha geniştir ve $\ell_1$ cezası girdi başına yalnızca birkaç birimin
aktif olmasını zorlar — umut, bu birimlerin altta yatan tek anlamlı (monosemantic) özniteliklere
karşılık gelmesidir.

Açık problemler gerçektir ve söylenmeye değer: $\lambda$, yeniden kurulumu seyreklikle ilkesel bir
ayar olmaksızın takas eder; sözlük genişliği değiştirilerek öznitelikler bölünebilir veya
birleşebilir; ve *iyi yeniden kurulum, doğruluğun kanıtı değildir*. Aşağıda bir SAE, rastgele
doğrusal bir karışımdan gerçek seyrek öznitelikleri geri kazanıyor; bu da geri kazanımı doğrudan
ölçmemize izin veriyor — gerçek bir ağda sahip olmadığımız bir lüks.


In [ ]:
rng = np.random.default_rng(5)
m_true, d_obs, n = 24, 16, 8000
D_true = rng.normal(size=(d_obs, m_true)); D_true /= np.linalg.norm(D_true, axis=0)
Z_true = (rng.random((n, m_true)) < 0.05)*rng.random((n, m_true))*2      # seyrek aktivasyonlar
A = Z_true @ D_true.T                                                    # gözlenen aktivasyonlar

def train_sae(A, m_dict, lam=0.2, steps=8000, lr=0.05, seed=0):
    r = np.random.default_rng(seed)
    We = r.normal(size=(A.shape[1], m_dict))*0.1
    Wd = r.normal(size=(m_dict, A.shape[1]))*0.1
    be, bd = np.zeros(m_dict), np.zeros(A.shape[1])
    for t in range(steps):
        idx = r.integers(0, len(A), 256)
        a = A[idx]
        pre = (a - bd)@We + be
        z = relu(pre)
        rec = z@Wd + bd
        err = rec - a
        gz = (err@Wd.T + lam*np.sign(z))*(pre > 0)
        Wd -= lr*(z.T@err)/len(a)
        We -= lr*((a-bd).T@gz)/len(a)
        be -= lr*gz.mean(0)
        bd -= lr*(err.mean(0) - (gz@We.T).mean(0))
        Wd /= np.maximum(np.linalg.norm(Wd, axis=1, keepdims=True), 1e-6)
    return We, Wd, be, bd

We, Wd, be, bd = train_sae(A, m_dict=64)
Z = relu((A - bd)@We + be)
rec = Z@Wd + bd

# Sözlük gerçek özniteliklerin kaçını geri kazanıyor?
cos = np.abs(Wd @ D_true) / (np.linalg.norm(Wd, axis=1, keepdims=True)*np.linalg.norm(D_true, axis=0)[None] + 1e-9)
best = cos.max(0)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
axes[0].hist(best, bins=20, color="steelblue", alpha=0.85)
axes[0].axvline(0.9, c="crimson", ls="--", lw=1.5, label="geri kazanım eşiği 0.9")
axes[0].set_xlabel("öğrenilen sözlük atomlarına maks kosinüs benzerliği")
axes[0].set_ylabel("gerçek öznitelik sayısı")
axes[0].set_title(f"{m_true} gerçek öznitelikten {(best > 0.9).sum()} tanesi geri kazanıldı")
axes[0].legend(fontsize=9)

axes[1].hist((Z > 1e-3).sum(1), bins=np.arange(0, 15)-0.5, color="seagreen", alpha=0.85, label="SAE kodu")
axes[1].hist((Z_true > 1e-3).sum(1), bins=np.arange(0, 15)-0.5, color="orange", alpha=0.6, label="gerçek değer")
axes[1].set_xlabel("örnek başına aktif birim"); axes[1].set_title("Öğrenilen kodun seyrekliği")
axes[1].legend(fontsize=9)

lams = [0.02, 0.05, 0.1, 0.2, 0.4]
recs, sps = [], []
for l in lams:
    We_, Wd_, be_, bd_ = train_sae(A, 64, lam=l, steps=3000)
    Z_ = relu((A-bd_)@We_ + be_)
    recs.append(np.mean((Z_@Wd_ + bd_ - A)**2)/np.mean(A**2))
    sps.append((Z_ > 1e-3).sum(1).mean())
axes[2].plot(sps, recs, "o-", lw=2)
for l, s_, r_ in zip(lams, sps, recs):
    axes[2].annotate(f"lam={l}", (s_, r_), fontsize=8, xytext=(4, 4), textcoords="offset points")
axes[2].set_xlabel("ortalama aktif birim"); axes[2].set_ylabel("relative reconstruction error")
axes[2].set_title("Uygulayıcının üzerinde seçim yapmak zorunda olduğu Pareto sınırı"); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"göreli yeniden kurulum hatası: {np.mean((rec-A)**2)/np.mean(A**2):.4f}")
print(f"örnek başına ortalama aktif birim: {(Z > 1e-3).sum(1).mean():.2f}  (gerçek {(Z_true > 1e-3).sum(1).mean():.2f})")


## 8. Çalışan Bir Yöntembilim

Mekanistik iddialar nedensel iddialardır; dolayısıyla nedensel kanıt gerektirirler:

| Teknik | Cevapladığı soru |
|---|---|
| **Aktivasyon yamalama** | Bir bileşenin aktivasyonunu başka bir girdininkiyle değiştir — çıktı onu takip ediyor mu? |
| **Ablasyon** | Bir başlığı/nöronu sıfırla veya ortalamayla değiştir; ilgili davranıştaki kaybı ölç |
| **Yol yamalama** | Etkiyi hangi *güzergâh* taşıyor, yalnızca hangi bileşen değil |
| **Logit merceği** | Ara artık akışlarını çıktı başlığıyla çöz ve bir tahminin nasıl oluştuğunu izle |
| **Devre analizi** | Parçaları bir mekanizmada birleştir, sonra *yeni* bir davranış öngör ve test et |

Bir hikâyeyi bir sonuçtan ayıran şey son satırdır. Bir yorumlanabilirlik iddiası ancak yanlışlanabilir
bir öngörü ürettiğinde — şu ablasyon şu davranışı bozacak, başka hiçbir şeyi bozmayacak — ve öngörü
ayakta kaldığında değerlidir.

## 9. Özet

| Kavram | Açıklama |
|---|---|
| **Gradyan saliency** | Katkıyı değil duyarlılığı ölçer; doyar ve süreksizdir |
| **Bütünleşik gradyanlar** | Taban çizgisinden yol integrali; bütünlüğü sağlar |
| **Taban çizgisi seçimi** | "Yokluk"un ne demek olduğunu tanımlar; sorunun bir parçasıdır |
| **Shapley değerleri** | 4 aksiyom altında tek atıf; $2^n$ maliyet; etkileşim kredisini böler |
| **Akıl sağlığı testleri** | Model/veri rastgeleleştirme; birçok saliency yöntemi kenar bulucudur |
| **Doğrusal sondalar** | Seçicilik için kontrol görevi, nedensellik için müdahale gerekir |
| **Süperpozisyon** | Seyrek öznitelikler az boyuta paketlenir ⇒ çok anlamlı nöronlar |
| **Seyrek otokodlayıcı** | Aşırı tam sözlük + $\ell_1$; geri kazanım, yeniden kurulumla kanıtlanmaz |
| **Nedensel yöntemler** | Yamalama, ablasyon, yol yamalama — bir mekanizma iddiasının gerektirdiği kanıt |

**Sonraki Defter →** Dayanıklılık, Düşmanca Örnekler ve Dağılım Kayması
